# ORIGINAL 03 (pre-refactor, reconstructed for reference)

This is the original untracked notebook, preserved before the package refactor. The maintained version is the thin-caller `notebooks/03_inferemce_engine.ipynb`.


In [ ]:
import pandas as pd
import joblib
from nba_api.stats.endpoints import leaguegamefinder, boxscoretraditionalv2, leaguedashplayerstats, leaguegamelog, scoreboardv2
from nba_api.stats.static import teams
import numpy as np
from datetime import datetime, timedelta

In [ ]:
model = joblib.load('nba_model_2024.pkl')
iso = joblib.load('iso_model.pkl')
diff_features = joblib.load('model_features.pkl')

In [ ]:
# Fetch games for the current 2025-26 season
finder = leaguegamefinder.LeagueGameFinder(season_nullable="2025-26", league_id_nullable="00", season_type_nullable="Regular Season")
games_2526 = finder.get_data_frames()[0]

# Fetch player stats to identify stars
players_stats = leaguedashplayerstats.LeagueDashPlayerStats(season="2025-26").get_data_frames()[0]
stars = players_stats.sort_values('PTS', ascending=False).drop_duplicates('TEAM_ID')
stars_list = stars[['TEAM_ID', 'PLAYER_ID', 'PLAYER_NAME']]

# Fetch player game logs to track active status
player_logs = leaguegamelog.LeagueGameLog(
    season='2025-26', 
    player_or_team_abbreviation='P'
).get_data_frames()[0]

star_attendance = player_logs[player_logs['PLAYER_ID'].isin(stars_list['PLAYER_ID'])][['GAME_ID', 'PLAYER_ID']]
star_attendance['IS_ACTIVE'] = 1

In [ ]:
games_2526['POSS'] = 0.96 * (games_2526['FGA'] + games_2526['TOV'] + 0.4 * games_2526['FTA'] - games_2526['OREB'])
games_2526['eFG%'] = (games_2526['FGM'] + 0.5 * games_2526['FG3M']) / games_2526['FGA']
games_2526['TS%'] = games_2526['PTS'] / (2 * games_2526['FGA'] + 0.44 * games_2526['FTA'])
games_2526['NET_RAT'] = 100 * (games_2526['PTS'] / games_2526['POSS'] - (games_2526['PTS'] - games_2526['PLUS_MINUS']) / games_2526['POSS'])

In [ ]:
games_2526['GAME_DATE'] = pd.to_datetime(games_2526['GAME_DATE'])
games_2526['CUM_WINS'] = games_2526.groupby('TEAM_ID')['WL'].transform(lambda x: (x == 'W').astype(int).cumsum().shift(1).fillna(0))
games_2526['GAMES_PLAYED'] = games_2526.groupby('TEAM_ID').cumcount().astype(int)
games_2526['SEASON_WIN_PCT'] = games_2526['CUM_WINS'] / games_2526['GAMES_PLAYED']
games_2526['SEASON_WIN_PCT'] = games_2526['SEASON_WIN_PCT'].fillna(0)

games_2526['DAYS_REST'] = games_2526.groupby('TEAM_ID')['GAME_DATE'].diff().dt.days
games_2526['DAYS_REST'] = games_2526['DAYS_REST'].clip(upper=4).fillna(4)
games_2526['IS_B2B'] = (games_2526['DAYS_REST'] <= 1).astype(int)

games_2526 = games_2526.dropna()
games_2526 = games_2526.sort_values(by=['TEAM_ID', 'GAME_DATE'])

cols_to_roll = ['AST', 'REB', 'TOV', 'TS%', 'NET_RAT', 'SEASON_WIN_PCT']
for col in cols_to_roll:
    games_2526['ROLL_' + col] = games_2526.groupby('TEAM_ID')[col].transform(lambda x: x.ewm(span=10).mean().shift(1))

team_abbr_to_id = {t['abbreviation']: t['id'] for t in teams.get_teams()}
games_2526['OPP_ABBR'] = games_2526['MATCHUP'].apply(lambda x: x.split(' ')[-1])
games_2526['OPP_ID'] = games_2526['OPP_ABBR'].map(team_abbr_to_id)

win_pct_map = games_2526.set_index(['TEAM_ID', 'GAME_DATE'])['SEASON_WIN_PCT'].to_dict()
games_2526['OPP_WIN_PCT_AT_GAME'] = games_2526.apply(lambda x: win_pct_map.get((x['OPP_ID'], x['GAME_DATE']), 0.5), axis=1)

games_2526['ROLL_SOS'] = games_2526.groupby('TEAM_ID')['OPP_WIN_PCT_AT_GAME'].transform(lambda x: x.rolling(10).mean().shift(1).fillna(0.5))
games_2526['ADJ_NET_RAT'] = games_2526['ROLL_NET_RAT'] * games_2526['ROLL_SOS']

# Fatigue calculations
games_2526 = games_2526.sort_values(['TEAM_ID', 'GAME_DATE']).reset_index(drop=True)
games_2526['GAMES_IN_LAST_4_DAYS'] = games_2526.groupby('TEAM_ID').apply(lambda x: x.set_index('GAME_DATE')['WL'].rolling('4D').count().shift(1).fillna(0), include_groups=False).values
games_2526['GAMES_IN_LAST_5_DAYS'] = games_2526.groupby('TEAM_ID').apply(lambda x: x.set_index('GAME_DATE')['WL'].rolling('5D').count().shift(1).fillna(0), include_groups=False).values
games_2526['FATIGUE_3_IN_4'] = (games_2526['GAMES_IN_LAST_4_DAYS'] >= 3).astype(int)
games_2526['FATIGUE_4_IN_5'] = (games_2526['GAMES_IN_LAST_5_DAYS'] >= 4).astype(int)

In [ ]:
stats_cols = ['GAME_ID', 'TEAM_NAME', 'TEAM_ID', 'WL', 'SEASON_WIN_PCT', 'DAYS_REST', 'GAMES_PLAYED', 'GAME_DATE', 'TS%', 'NET_RAT', 'IS_B2B', 'FATIGUE_3_IN_4', 'FATIGUE_4_IN_5', 'GAMES_IN_LAST_4_DAYS', 'GAMES_IN_LAST_5_DAYS', 'ADJ_NET_RAT'] + [col for col in games_2526.columns if 'ROLL_' in col]

home_games = games_2526[games_2526['MATCHUP'].str.contains('vs')][stats_cols].copy()
away_games = games_2526[games_2526['MATCHUP'].str.contains(' @ ')][stats_cols].copy()

home_games = home_games.add_prefix('HOME_').rename(columns={'HOME_GAME_ID': 'GAME_ID', 'HOME_GAME_DATE': 'GAME_DATE'})
away_games = away_games.add_prefix('AWAY_').rename(columns={'AWAY_GAME_ID': 'GAME_ID'})

model_df = pd.merge(home_games, away_games, on='GAME_ID')
model_df['HOME_WIN'] = (model_df['HOME_WL'] == 'W').astype(int)

In [ ]:
star_map = dict(zip(stars_list['TEAM_ID'], stars_list['PLAYER_ID']))
model_df['HOME_STAR_ID'] = model_df['HOME_TEAM_ID'].map(star_map)
model_df['AWAY_STAR_ID'] = model_df['AWAY_TEAM_ID'].map(star_map)
active_pairs = set(zip(star_attendance['GAME_ID'], star_attendance['PLAYER_ID']))

for col in ['AST', 'REB', 'TOV', 'TS%', 'NET_RAT']:
    model_df[f'DIFF_{col}'] = model_df[f'HOME_ROLL_{col}'] - model_df[f'AWAY_ROLL_{col}']

model_df['DIFF_SEASON_WIN_PCT'] = model_df['HOME_SEASON_WIN_PCT'] - model_df['AWAY_SEASON_WIN_PCT']
model_df['DIFF_REST'] = model_df['HOME_DAYS_REST'] - model_df['AWAY_DAYS_REST']
model_df['DIFF_STAR_ACTIVE'] = [
    (1 if (game, h_star) in active_pairs else 0) - (1 if (game, a_star) in active_pairs else 0)
    for game, h_star, a_star in zip(model_df['GAME_ID'], model_df['HOME_STAR_ID'], model_df['AWAY_STAR_ID'])
]
model_df['DIFF_B2B'] = model_df['HOME_IS_B2B'] - model_df['AWAY_IS_B2B']
model_df['DIFF_GAMES_4D'] = model_df['HOME_GAMES_IN_LAST_4_DAYS'] - model_df['AWAY_GAMES_IN_LAST_4_DAYS']
model_df['DIFF_GAMES_5D'] = model_df['HOME_GAMES_IN_LAST_5_DAYS'] - model_df['AWAY_GAMES_IN_LAST_5_DAYS']
model_df['DIFF_3_IN_4'] = model_df['HOME_FATIGUE_3_IN_4'] - model_df['AWAY_FATIGUE_3_IN_4']
model_df['DIFF_4_IN_5'] = model_df['HOME_FATIGUE_4_IN_5'] - model_df['AWAY_FATIGUE_4_IN_5']
model_df['DIFF_ADJ_NET_RAT'] = model_df['HOME_ADJ_NET_RAT'] - model_df['AWAY_ADJ_NET_RAT']

model_df['HOME_FATIGUE_PENALTY'] = (1 - 0.12 * model_df['HOME_IS_B2B'] - 0.08 * model_df['HOME_FATIGUE_3_IN_4'] - 0.05 * model_df['HOME_FATIGUE_4_IN_5'] - 0.05 * model_df['HOME_GAMES_IN_LAST_5_DAYS'].clip(lower=2) / 3)
model_df['AWAY_FATIGUE_PENALTY'] = (1 - 0.12 * model_df['AWAY_IS_B2B'] - 0.08 * model_df['AWAY_FATIGUE_3_IN_4'] - 0.05 * model_df['AWAY_FATIGUE_4_IN_5'] - 0.05 * model_df['AWAY_GAMES_IN_LAST_5_DAYS'].clip(lower=2) / 3)
model_df['DIFF_FATIGUE_ADJ_NET_RAT'] = (model_df['HOME_ADJ_NET_RAT'] * model_df['HOME_FATIGUE_PENALTY'] - model_df['AWAY_ADJ_NET_RAT'] * model_df['AWAY_FATIGUE_PENALTY'])

model_df['GAME_DATE'] = pd.to_datetime(model_df['GAME_DATE'])
model_df = model_df.sort_values('GAME_DATE').reset_index(drop=True)

In [ ]:
X_2526 = model_df[diff_features]
y_actual = model_df['HOME_WIN']

preds_2526 = model.predict(X_2526)
probs_2526 = model.predict_proba(X_2526)[:, 1]

print(f"Current Season Prediction Accuracy: {(preds_2526 == y_actual).mean():.2%}")

In [ ]:
results_2526 = X_2526.copy()
results_2526['GAME_DATE'] = model_df['GAME_DATE']
results_2526['HOME_TEAM'] = model_df['HOME_TEAM_NAME']
results_2526['AWAY_TEAM'] = model_df['AWAY_TEAM_NAME']
results_2526['ACTUAL'] = y_actual
results_2526['PRED'] = preds_2526
results_2526['CONFIDENCE'] = probs_2526

# Filter for high confidence bets
high_conf = results_2526[results_2526['CONFIDENCE'] > 0.60]
if len(high_conf) > 0:
    print(f"Accuracy when >60% sure: {len(high_conf[high_conf['ACTUAL'] == high_conf['PRED']]) / len(high_conf):.2%}")
else:
    print("No high confidence bets found yet.")

# Show recent high confidence bets
print("\nRecent High Confidence Bets:")
print(high_conf[['GAME_DATE', 'HOME_TEAM', 'AWAY_TEAM', 'PRED', 'CONFIDENCE', 'ACTUAL']].tail(10))

In [ ]:
# --- FUTURE PREDICTION ENGINE ---

# Calculate latest rolling stats for all teams (for the NEXT game)
latest_rolling_stats = {}
cols_to_roll = ['AST', 'REB', 'TOV', 'TS%', 'NET_RAT', 'SEASON_WIN_PCT']

for team_id in games_2526['TEAM_ID'].unique():
    team_games = games_2526[games_2526['TEAM_ID'] == team_id].sort_values('GAME_DATE')
    team_stats = {}
    for col in cols_to_roll:
        # Calculate EWMA including the last game
        ewma = team_games[col].ewm(span=10).mean()
        team_stats[f'ROLL_{col}'] = ewma.iloc[-1]
    
    # Calculate ROLL_SOS
    opp_win_pcts = team_games['OPP_WIN_PCT_AT_GAME']
    team_stats['ROLL_SOS'] = opp_win_pcts.rolling(10).mean().iloc[-1]
    
    # Other stats
    team_stats['LAST_GAME_DATE'] = team_games['GAME_DATE'].iloc[-1]
    team_stats['RECENT_GAMES'] = team_games['GAME_DATE'].tolist()[-5:] # Last 5 game dates
    
    latest_rolling_stats[team_id] = team_stats

print("Latest team stats calculated for future predictions.")

In [ ]:
def predict_game(home_id, away_id, game_date_str):
    game_date = pd.to_datetime(game_date_str)
    
    home_stats = latest_rolling_stats.get(home_id)
    away_stats = latest_rolling_stats.get(away_id)
    
    if not home_stats or not away_stats:
        return None
        
    # Calculate Diffs
    features = {}
    for col in ['AST', 'REB', 'TOV', 'TS%', 'NET_RAT']:
        features[f'DIFF_{col}'] = home_stats[f'ROLL_{col}'] - away_stats[f'ROLL_{col}']
        
    features['DIFF_SEASON_WIN_PCT'] = home_stats['ROLL_SEASON_WIN_PCT'] - away_stats['ROLL_SEASON_WIN_PCT']
    
    # Rest & Fatigue
    home_days_rest = (game_date - home_stats['LAST_GAME_DATE']).days
    away_days_rest = (game_date - away_stats['LAST_GAME_DATE']).days
    features['DIFF_REST'] = home_days_rest - away_days_rest
    
    home_is_b2b = 1 if home_days_rest <= 1 else 0
    away_is_b2b = 1 if away_days_rest <= 1 else 0
    features['DIFF_B2B'] = home_is_b2b - away_is_b2b
    
    # Fatigue (Games in last X days)
    def count_games_in_window(dates, target_date, days):
        cutoff = target_date - pd.Timedelta(days=days)
        return sum(1 for d in dates if d > cutoff and d < target_date)

    h_g4 = count_games_in_window(home_stats['RECENT_GAMES'], game_date, 4)
    a_g4 = count_games_in_window(away_stats['RECENT_GAMES'], game_date, 4)
    features['DIFF_GAMES_4D'] = h_g4 - a_g4
    
    h_g5 = count_games_in_window(home_stats['RECENT_GAMES'], game_date, 5)
    a_g5 = count_games_in_window(away_stats['RECENT_GAMES'], game_date, 5)
    features['DIFF_GAMES_5D'] = h_g5 - a_g5
    
    h_3in4 = 1 if h_g4 >= 3 else 0
    a_3in4 = 1 if a_g4 >= 3 else 0
    features['DIFF_3_IN_4'] = h_3in4 - a_3in4
    
    h_4in5 = 1 if h_g5 >= 4 else 0
    a_4in5 = 1 if a_g5 >= 4 else 0
    features['DIFF_4_IN_5'] = h_4in5 - a_4in5
    
    # Stars Active (Assume active for now)
    features['DIFF_STAR_ACTIVE'] = 0 
    
    # Adj Net Rat
    # Calculate ADJ_NET_RAT for home and away separately first
    home_adj_net = home_stats['ROLL_NET_RAT'] * home_stats['ROLL_SOS']
    away_adj_net = away_stats['ROLL_NET_RAT'] * away_stats['ROLL_SOS']
    features['DIFF_ADJ_NET_RAT'] = home_adj_net - away_adj_net
    
    # Fatigue Penalty
    h_fatigue_penalty = (1 - 0.12 * home_is_b2b - 0.08 * h_3in4 - 0.05 * h_4in5 - 0.05 * max(h_g5, 2) / 3)
    a_fatigue_penalty = (1 - 0.12 * away_is_b2b - 0.08 * a_3in4 - 0.05 * a_4in5 - 0.05 * max(a_g5, 2) / 3)
    
    features['DIFF_FATIGUE_ADJ_NET_RAT'] = (home_adj_net * h_fatigue_penalty - away_adj_net * a_fatigue_penalty)
    
    # Create DataFrame
    df = pd.DataFrame([features])
    
    # Ensure all columns are present
    for col in diff_features:
        if col not in df.columns:
            df[col] = 0
            
    # Predict
    pred = model.predict(df[diff_features])[0]
    prob = model.predict_proba(df[diff_features])[0][1]
    
    return {
        "Home_Win_Prob": prob,
        "Prediction": "Home Win" if pred == 1 else "Away Win"
    }

print("Prediction function ready.")

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
from nba_api.stats.endpoints import scoreboardv2

# =========================================================
# CONFIGURATION
# =========================================================
MIN_EDGE = 0.03          # minimum edge vs market prob
KELLY_CAP = 0.20        # max 3% bankroll per bet
BANKROLL = 500.0         # starting bankroll for calculation

# =========================================================
# FUNCTION: SUGGEST BET AMOUNT BASED ON MODEL CONFIDENCE
# =========================================================
def suggest_bet_amount(model_prob, decimal_odds, bankroll, kelly_cap,kelly_multiplier=0.5):
    """
    Suggests a bet amount using fractional Kelly Criterion.
    """
    if not (0 < model_prob < 1):
        raise ValueError("Model probability must be between 0 and 1.")
    if decimal_odds <= 1:
        raise ValueError("Decimal odds must be > 1.")
    if model_prob <= 0.60:
        return 0.0
    b = decimal_odds - 1
    p = model_prob
    q = 1 - p
    
    # Kelly fraction
    f = ((b * p) - q) / b * kelly_multiplier
    f = max(0.0, f)        # no negative bet
    f = min(f, kelly_cap)  # cap the fraction
    
    stake = bankroll * f
    return stake

# =========================================================
# FUNCTION: GET TOMORROW'S GAMES
# =========================================================
def get_tomorrows_games():
    date = (datetime.now()).strftime('%Y-%m-%d')
    print(f"Fetching schedule for {date}...")

    board = scoreboardv2.ScoreboardV2(game_date=date)
    games = board.game_header.get_data_frame()

    if games.empty:
        return pd.DataFrame()

    games = games[games['GAME_STATUS_ID'] == 1].copy()  # only scheduled games
    games['GAME_DATE_STR'] = date
    return games

# =========================================================
# EXECUTION: BETTING CALCULATOR LOOP
# =========================================================
next_games = get_tomorrows_games()

if next_games.empty:
    print("No games scheduled for tomorrow.")
else:
    for _, game in next_games.iterrows():
        home_id = game['HOME_TEAM_ID']
        away_id = game['VISITOR_TEAM_ID']
        game_date = game['GAME_DATE_STR']

        home_row = games_2526[games_2526['TEAM_ID'] == home_id]
        away_row = games_2526[games_2526['TEAM_ID'] == away_id]

        if home_row.empty or away_row.empty:
            continue

        home_name = home_row.iloc[-1]['TEAM_NAME']
        away_name = away_row.iloc[-1]['TEAM_NAME']

        result = predict_game(home_id, away_id, game_date)
        if not result:
            continue

        # Determine model pick and confidence
        if result['Prediction'] == 'Home Win':
            pick = home_name
            conf = result['Home_Win_Prob']
        else:
            pick = away_name
            conf = 1 - result['Home_Win_Prob']

        print(f"\nMatchup: {away_name} @ {home_name}")
        print(f"Model Pick: {pick} (Model Confidence: {conf:.1%})")

        # --- Input decimal odds directly ---
        odds_in = input(f"Enter decimal odds for {pick} (Enter to skip): ").strip()
        if not odds_in:
            continue

        try:
            odds = float(odds_in)
        except ValueError:
            print("Invalid odds. Skipping.")
            continue

        if odds <= 1.0:
            print("Odds must be > 1. Skipping.")
            continue

        # --- Calculate metrics ---
        implied_prob = 1 / odds
        edge = conf - implied_prob
        suggested_stake = suggest_bet_amount(conf, odds, BANKROLL, KELLY_CAP)
        ev = edge * suggested_stake

        # --- Display metrics ---
        print(f"Decimal Odds: {odds:.2f} → Implied Probability: {implied_prob:.2%}")
        print(f"Edge vs Market: {edge:.1%}")
        print(f"Suggested Stake (Kelly): ${suggested_stake:.2f}")
        print(f"Expected Value: ${ev:.2f}")

        # --- Edge filter ---
        if edge >= MIN_EDGE:
            print(f"Edge ≥ {MIN_EDGE:.0%} → potential bet!")
        else:
            print(f"Edge below {MIN_EDGE:.0%} → skip")